In [35]:
using LowLevelFEM

## 2D steady-state heat conduction with convection

This example solves a steady-state heat conduction problem on a rectangular 2D domain.  
The left boundary has a prescribed temperature, while the right boundary is exposed to convective heat transfer.

The purpose of the example is twofold:

1. solve the problem using the built-in heat conduction interface;
2. reproduce the same system using the weak-form operator notation.

First, a simple rectangular mesh is generated and a heat conduction problem is created.

The material is assigned to the physical group `"body"`.  
For a plane heat conduction problem the unknown field is the temperature $T$, with one degree of freedom per node.

In [36]:
structured_rect_mesh()

mat = Material("body")
prob = Problem([mat], type=:PlaneHeatConduction);

Two boundary conditions are defined:

- on the left boundary, the temperature is prescribed as $T = 20$;
- on the right boundary, convection is applied with heat transfer coefficient \(h = 100\) and ambient temperature $T_\infty = 40$.

The convective boundary condition corresponds to

$$
q_n = h (T_\infty - T),
$$

where $q_n$ is the heat flux entering the body through the boundary.

In [37]:
bc1 = temperatureConstraint("left", T=20)
bc2 = heatConvection("right", h=100, T∞=40);

The temperature field is solved using the high-level heat conduction solver.

Internally, the solver assembles the heat conduction matrix, adds the convection contribution on the boundary, applies the prescribed temperature condition, and solves the resulting linear system.

In [38]:
T1 = solveTemperature(prob, temperatureConstraint=[bc1], heatConvection=[bc2]);

"C0 = sparse([2, 14, 3, 22, 2, 14, 15, 14, 15, 16, 15, 16, 17, 16, 17, 18, 17, 18, 19, 18, 19, 20, 19, 20, 21, 20, 21, 22, 3, 21, 22], [2, 2, 3, 3, 14, 14, 14, 15, 15, 15, 16, 16, 16, 17, 17, 17, 18, 18, 18, 19, 19, 19, 20, 20, 20, 21, 21, 21, 22, 22, 22], [3.333333333333" ⋯ 329 bytes ⋯ "667, 1.6666666666666683, 1.6666666666666683, 6.66666666666667, 1.6666666666666665, 1.6666666666666665, 6.666666666666666, 1.6666666666666665, 1.6666666666666665, 6.666666666666666, 1.6666666666666665, 1.6666666666666665, 1.6666666666666665, 6.666666666666666], 121, 121)"

After the temperature field has been computed, the heat flux is evaluated from Fourier's law,

$$
\mathbf{q} = -k \nabla T .
$$

The result can be visualized together with the temperature field in the Gmsh post-processor.

In [39]:
q1 = solveHeatFlux(T1);

In [40]:
showDoFResults(T1, name="T1")
showHeatFluxResults(q1, name="q1");

## Weak-form formulation

The same problem can also be assembled explicitly using the weak-form operator interface.

A scalar-field problem is introduced for the temperature field.  
This makes it possible to write the finite element matrices directly from the weak form.

In [41]:
PT = Problem([mat], type=:ScalarField, dim=2, field=:T, rhs_field=:q);

In [42]:
k = mat.k
ρ = mat.ρ
c = mat.c;

The conductivity $k$ is taken from the material definition.

For steady-state heat conduction without internal heat generation, the domain contribution is

$$
\int_\Omega \nabla \delta T \cdot k \nabla T \, d\Omega .
$$

In the code this is written as

$$
\int (\mathrm{Grad}(T) \cdot k \cdot \mathrm{Grad}(T)).
$$

In [43]:
K = ∫(Grad(PT) ⋅ k ⋅ Grad(PT))
K[:, :]

121×121 SparseArrays.SparseMatrixCSC{Float64, Int64} with 961 stored entries:
⎡⡑⢌⡀⠀⠒⠀⠀⠂⠀⠀⠄⠀⠀⡁⠀⠀⡄⠀⠀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠂⠀⠐⎤
⎢⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠁⠀⠀⠃⠀⠀⠃⠀⠀⠆⠀⠀⡆⠀⠀⡄⠀⠀⡀⠀⠀⡀⠀⠀⎥
⎢⠘⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠁⠀⠀⢷⣄⠀⎥
⎢⠠⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠀⠀⢠⠀⠙⢷⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⡀⠀⠀⡄⠀⢠⠀⠀⠰⠀⠀⠸⠀⠀⠘⠀⠀⠈⠀⠀⠈⠀⠀⠀⎥
⎢⠀⠁⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⣠⡾⠃⠀⠀⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠄⠠⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⣻⣾⡋⠀⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣠⡾⠋⠈⠻⣦⡈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠉⠁⠀⠀⠀⠀⠀⠀⠈⠉⠀⠀⠻⣦⡈⠻⣦⡈⠻⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠠⠤⠀⠀⠀⠀⠀⠀⠤⠄⠀⠀⠀⠈⠻⣦⡈⠻⣦⡈⠻⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⣀⠀⠀⠀⠀⠀⠀⠀⠛⣦⡈⠻⣦⠈⠻⣦⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠻⣦⡀⠻⣦⡀⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠠⠄⠀⠀⠀⠀⠐⠂⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⣤⡈⠛⣤⡈⠛⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⠈⠻⣦⠈⠻⣦⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠈⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠻⣦⡀⠻⣦⡀⠻⣦⡀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠤⠀⠀⠀⠀⠒⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⣤⡈⠛⣤⡈⠓⣄⡀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⢀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⢦⠈⠻⣦⠈⠳⣦⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⢦⡀⠻⣦⡀⠻⣦⡀⎥
⎢⠠⠀⠀⠠⢤⣄⠀⠒⠂⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⣤⡈⠛⣤⡈⠛⎥
⎣⢀⠀⠀⠀⠀⠙⢷⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⠈⠻⣦⎦

## Convection boundary contribution

The convection boundary condition is

$$
q_n = h(T_\infty - T).
$$

In weak form, this gives the additional boundary terms

$$
q_n\Rightarrow -\int_{\Gamma_c} \delta T \, h T \, d\Gamma
+
\int_{\Gamma_c} \delta T \, h T_\infty \, d\Gamma .
$$

Therefore, convection contributes both to the left-hand side and to the right-hand side:

$$
K_c = \int_{\Gamma_c} N^T h N \, d\Gamma,
$$

$$
f_c = \int_{\Gamma_c} N^T h T_\infty \, d\Gamma.
$$

In the example, the convection boundary is the physical group `"right"`.

In [44]:
h=100

Kc = ∫(PT ⋅ h ⋅ PT, Γ="right")
Kc[:, :]

121×121 SparseArrays.SparseMatrixCSC{Float64, Int64} with 31 stored entries:
⎡⠐⠀⠀⠀⠐⠀⠀⠂⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠐⠀⠀⠀⠰⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠠⠀⠀⠀⠀⠈⠻⠆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

The vector `qc` contains the right-hand-side contribution caused by the ambient temperature.

Physically, this term represents the heat supplied by the surrounding environment.  
If $T_\infty$ is higher than the body temperature, heat enters the domain; if it is lower, heat leaves the domain.

In [45]:
qc = ∫(PT ⋅ (h * 40), Γ="right");

There is no internal heat source in this example, therefore the domain load vector is zero:

$$
f_\Omega = \int_\Omega N^T \cdot 0 \, d\Omega .
$$

The total right-hand side is the sum of the internal source term and the convection load term.

In [46]:
q = ∫(PT ⋅ 0);

The prescribed temperature on the left boundary is applied as a Dirichlet boundary condition.

In [47]:
bc = BoundaryCondition("left", T=20);

The final linear system is

$$
(K + K_c) T = f_\Omega + f_c .
$$

This is the explicit weak-form equivalent of the high-level call used earlier.

In [48]:
T2 = solveField(K + Kc, q + qc, support=[bc]);

Finally, the heat flux is computed directly from the temperature gradient:

$$
\mathbf{q} = -k \nabla T .
$$

This result should match the heat flux obtained from the high-level heat conduction solver.

In [49]:
q2 = -k * grad(T2);

The two solutions, `T1` and `T2`, are obtained through different interfaces, but they represent the same finite element problem:

- `T1` is computed using the built-in heat conduction solver;
- `T2` is computed from explicitly assembled weak-form terms.

This comparison is useful for checking that the convection boundary contribution has been assembled consistently.

In [50]:
showDoFResults(T2, name="T2")
showElementResults(q2, name="q2")

3

In [51]:
openPostProcessor()